[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_head = d_model // num_heads
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_model = d_model
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head)
        self.W_o = nn.Linear(d_model, d_model)



    def forward(self, x):
        *B, S, _ = x.shape
        Q = self.W_q(x).view(*B, S, self.num_heads, self.d_head).transpose(-2, -3)
        K = self.W_k(x).view(*B, S, self.num_kv_heads, self.d_head).transpose(-2, -3)
        V = self.W_v(x).view(*B, S, self.num_kv_heads, self.d_head).transpose(-2, -3)
        scale = self.d_head ** 0.5
        repeat = self.num_heads // self.num_kv_heads
        K = K.repeat_interleave(repeat, dim=-3)
        V = V.repeat_interleave(repeat, dim=-3)
        scores = torch.einsum('...qd,...kd->...qk', Q, K) / scale
        atten_weight = torch.softmax(scores, dim=-1)
        atten = torch.einsum('...qk,...kd->...qd', atten_weight, V)
        output = atten.transpose(-2, -3).reshape(*B, S, self.d_model)
        return self.W_o(output)


In [9]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [ ]:
from torch_judge import check
check('gqa')